In [1]:
from gitsource import GithubRepositoryDataReader, chunk_documents

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

documents = [file.parse() for file in reader.read()]

len(documents)

72

In [2]:
chunks = chunk_documents(documents, size=2000, step=1000)

len(chunks)

295

In [1]:
import json
from pydantic import BaseModel
from dotenv import load_dotenv
from openai import OpenAI

from evaluation_utils import llm_structured

load_dotenv(override=True)
openai_client = OpenAI()


class Questions(BaseModel):
    questions: list[str]


data_gen_instructions = """
You emulate a student who is taking our LLM course.
You are given one lesson page from the course.
Formulate 5 questions this student might ask that are answered by this page.

Rules:
- The page should contain the answer to each question.
- Make the questions complete and not too short.
- Use as few words as possible from the page; don't copy its phrasing.
- The questions should resemble how people actually ask things online:
  not too formal, not too short, not too long.
- Ask about the content of the lesson, not about its formatting or filename.
""".strip()

In [2]:
records = []
usages = []

for doc in documents[:3]:
    user_prompt = json.dumps({
        "filename": doc["filename"],
        "content": doc["content"],
    })

    result, usage = llm_structured(
        openai_client,
        data_gen_instructions,
        user_prompt,
        Questions
    )

    usages.append(usage)

    for q in result.questions:
        records.append({
            "question": q,
            "filename": doc["filename"],
        })

input_tokens = [u.input_tokens for u in usages]

input_tokens, sum(input_tokens) / len(input_tokens)

NameError: name 'documents' is not defined

In [3]:
from gitsource import GithubRepositoryDataReader, chunk_documents

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

documents = [file.parse() for file in reader.read()]

len(documents)

72

In [4]:
chunks = chunk_documents(documents, size=2000, step=1000)

len(chunks)

295

In [5]:
import json
from pydantic import BaseModel
from dotenv import load_dotenv
from openai import OpenAI

from evaluation_utils import llm_structured

load_dotenv(override=True)
openai_client = OpenAI()


class Questions(BaseModel):
    questions: list[str]


data_gen_instructions = """
You emulate a student who is taking our LLM course.
You are given one lesson page from the course.
Formulate 5 questions this student might ask that are answered by this page.

Rules:
- The page should contain the answer to each question.
- Make the questions complete and not too short.
- Use as few words as possible from the page; don't copy its phrasing.
- The questions should resemble how people actually ask things online:
  not too formal, not too short, not too long.
- Ask about the content of the lesson, not about its formatting or filename.
""".strip()

In [6]:
records = []
usages = []

for doc in documents[:3]:
    user_prompt = json.dumps({
        "filename": doc["filename"],
        "content": doc["content"],
    })

    result, usage = llm_structured(
        openai_client,
        data_gen_instructions,
        user_prompt,
        Questions
    )

    usages.append(usage)

    for q in result.questions:
        records.append({
            "question": q,
            "filename": doc["filename"],
        })

input_tokens = [u.input_tokens for u in usages]

input_tokens, sum(input_tokens) / len(input_tokens)

([1022, 1288, 1755], 1355.0)

In [7]:
import pandas as pd

df_ground_truth = pd.read_csv("data/ground-truth.csv")
ground_truth = df_ground_truth.to_dict(orient="records")

df_ground_truth.head(), len(ground_truth)

(                                            question  \
 0  What exactly is a retrieval-augmented generati...   
 1  Why does this course build the RAG project in ...   
 2  What are the main weaknesses of large language...   
 3  What will the course build in the first part o...   
 4  What kind of example app are you building here...   
 
                              filename  
 0  01-agentic-rag/lessons/01-intro.md  
 1  01-agentic-rag/lessons/01-intro.md  
 2  01-agentic-rag/lessons/01-intro.md  
 3  01-agentic-rag/lessons/01-intro.md  
 4  01-agentic-rag/lessons/01-intro.md  ,
 360)

In [8]:
from minsearch import Index

text_index = Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)

text_index.fit(chunks)


def text_search(query, num_results=5):
    return text_index.search(
        query=query,
        num_results=num_results
    )

In [9]:
q = ground_truth[0]["question"]

text_results = text_search(q)

q, text_results[0]["filename"]

("What exactly is a retrieval-augmented generation system, and why does it help with answers that the model wouldn't know on its own?",
 '01-agentic-rag/lessons/03-rag.md')

In [10]:
from embedder import Embedder
from minsearch import VectorSearch
import numpy as np

embedder = Embedder()

texts = [chunk["content"] for chunk in chunks]
X = np.array([embedder.encode(text) for text in texts])

vector_index = VectorSearch(keyword_fields=["filename"])
vector_index.fit(X, chunks)


def vector_search(query, num_results=5):
    v_query = embedder.encode(query)

    return vector_index.search(
        v_query,
        num_results=num_results
    )

In [11]:
vector_results = vector_search(q)

q, vector_results[0]["filename"]

("What exactly is a retrieval-augmented generation system, and why does it help with answers that the model wouldn't know on its own?",
 '01-agentic-rag/lessons/01-intro.md')

In [12]:
from tqdm.auto import tqdm


def compute_relevance(q, search_function):
    correct_filename = q["filename"]
    results = search_function(q["question"])

    relevance = []

    for d in results:
        relevance.append(int(d["filename"] == correct_filename))

    return relevance


def compute_relevance_total(ground_truth, search_function):
    relevance_total = []

    for q in tqdm(ground_truth):
        relevance = compute_relevance(q, search_function)
        relevance_total.append(relevance)

    return relevance_total


def hit_rate(relevance):
    cnt = 0

    for line in relevance:
        if 1 in line:
            cnt = cnt + 1

    return cnt / len(relevance)


def mrr(relevance):
    total_score = 0.0

    for line in relevance:
        for rank in range(len(line)):
            if line[rank] == 1:
                total_score = total_score + 1 / (rank + 1)
                break

    return total_score / len(relevance)


def evaluate(ground_truth, search_function):
    relevance_total = compute_relevance_total(ground_truth, search_function)

    return {
        "hit_rate": hit_rate(relevance_total),
        "mrr": mrr(relevance_total),
    }

In [13]:
text_eval = evaluate(ground_truth, text_search)

text_eval

  0%|          | 0/360 [00:00<?, ?it/s]

{'hit_rate': 0.7583333333333333, 'mrr': 0.5942592592592594}

In [14]:
vector_eval = evaluate(ground_truth, vector_search)

vector_eval

  0%|          | 0/360 [00:00<?, ?it/s]

{'hit_rate': 0.725, 'mrr': 0.5486111111111112}

In [15]:
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)

    return [docs[key] for key in ranked[:num_results]]


def hybrid_search(query, k=60):
    text_results = text_search(query, num_results=10)
    vector_results = vector_search(query, num_results=10)

    return rrf([text_results, vector_results], k=k)

In [16]:
hybrid_results = []

for k in [1, 50, 100, 200]:
    result = evaluate(
        ground_truth,
        lambda query, k=k: hybrid_search(query, k=k)
    )

    hybrid_results.append({
        "k": k,
        "hit_rate": result["hit_rate"],
        "mrr": result["mrr"],
    })

df_hybrid = pd.DataFrame(hybrid_results)

df_hybrid.sort_values("mrr", ascending=False)

  0%|          | 0/360 [00:00<?, ?it/s]

  0%|          | 0/360 [00:00<?, ?it/s]

  0%|          | 0/360 [00:00<?, ?it/s]

  0%|          | 0/360 [00:00<?, ?it/s]

,k,hit_rate,mrr
0,1,0.838889,0.648194
1,50,0.836111,0.637917
2,100,0.836111,0.637917
3,200,0.836111,0.637917
